In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy.optimize import linprog
from sklearn.base import clone
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import gaussian_kde
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.svm import SVC
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.metrics import (
    confusion_matrix, f1_score, precision_score, recall_score,
    accuracy_score, roc_auc_score, roc_curve
)
from itertools import product


# Plot style
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = "DejaVu Sans"
sns.set_style("whitegrid")

COLORS = {"B": "#5C9EE0", "M": "#E05C5C"}
C_B   = COLORS["B"] 
C_M   = COLORS["M"]  
C_AMB = "#F4A261"     
C_NEU = "#546E7A"     

def savefig(name, dpi=150):
    path = f"{name}.png"
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close("all")
    print(f"  Saved")
 
def pretty(s):
    return s.replace("_", " ").title()



    

In [2]:
# Load and clean data

df = pd.read_csv('Data for Task 1.csv')
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
df_clean = df.copy()
df_clean = df_clean.drop(columns=["id"])
df_clean["diagnosis"] = df_clean["diagnosis"].map({"M": 1, "B": 0})


# Define full dataset

all_features = [col for col in df_clean.columns if col != "diagnosis"]
y = df_clean["diagnosis"]
features_30 = df_clean[all_features]


# Summary

print("DATASET SETUP COMPLETE")
print(f"  Dataset           : {df.shape[0]} samples x {len(all_features)} original features")
print(f"  Full feature set  : {len(features_30)} features")
print(f"  Class balance     : {(y == 0).sum()} benign | {(y == 1).sum()} malignant")




DATASET SETUP COMPLETE
  Dataset           : 569 samples x 30 original features
  Full feature set  : 569 features
  Class balance     : 357 benign | 212 malignant


In [3]:
DROP_FEATURES = [
    # Cluster A (size) - dropped in all three statistic groups
    "area_mean",  "perimeter_mean",
    "area_se",    "perimeter_se",
    "area_worst", "perimeter_worst",
    # Cluster B (shape) - only concavity dropped; compactness retained
    "concavity_mean", "concavity_se", "concavity_worst",
]
 
features_21 = [f for f in all_features if f not in DROP_FEATURES]
 
print("Apply consistently across mean / se / worst")
print(f"  Dropped ({len(DROP_FEATURES)}): {DROP_FEATURES}")
print(f"  Final feature count: {len(features_21)}")
print("\n  Final 21 features:")
for f in features_21:
    print(f"    {f}")

Apply consistently across mean / se / worst
  Dropped (9): ['area_mean', 'perimeter_mean', 'area_se', 'perimeter_se', 'area_worst', 'perimeter_worst', 'concavity_mean', 'concavity_se', 'concavity_worst']
  Final feature count: 21

  Final 21 features:
    radius_mean
    texture_mean
    smoothness_mean
    compactness_mean
    concave points_mean
    symmetry_mean
    fractal_dimension_mean
    radius_se
    texture_se
    smoothness_se
    compactness_se
    concave points_se
    symmetry_se
    fractal_dimension_se
    radius_worst
    texture_worst
    smoothness_worst
    compactness_worst
    concave points_worst
    symmetry_worst
    fractal_dimension_worst


In [4]:
# MODELLING WITH BALANCED CLASS WEIGHTS
# Using final 21-feature set

X = df[features_21]
y = df["diagnosis"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
 
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
 
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
lr = LogisticRegression(max_iter=10000, random_state=42, class_weight="balanced")
lr.fit(X_train_sc, y_train)
 
y_pred = lr.predict(X_test_sc)
cm = confusion_matrix(y_test, y_pred)
cv_f1 = cross_val_score(lr, X_train_sc, y_train, cv=cv, scoring="f1").mean()

In [5]:
# INITIAL MODELING


cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

def evaluate_model(name, model, Xtr, Xte, use_sample_weight=False):

    if use_sample_weight:
        sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
        model.fit(Xtr, y_train, sample_weight=sample_weight)
    else:
        model.fit(Xtr, y_train)

    y_pred = model.predict(Xte)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(Xte)[:, 1]
    else:
        y_prob = model.decision_function(Xte)

    cm = confusion_matrix(y_test, y_pred, labels=["B", "M"])

    cv_scores = []

    for train_idx, val_idx in cv.split(Xtr, y_train):
        model_cv = clone(model)

        if isinstance(Xtr, np.ndarray):
            X_fold_train = Xtr[train_idx]
            X_fold_val = Xtr[val_idx]
        else:
            X_fold_train = Xtr.iloc[train_idx]
            X_fold_val = Xtr.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        if use_sample_weight:
            fold_weight = compute_sample_weight(
                class_weight="balanced",
                y=y_fold_train
            )
            model_cv.fit(X_fold_train, y_fold_train, sample_weight=fold_weight)
        else:
            model_cv.fit(X_fold_train, y_fold_train)

        y_fold_pred = model_cv.predict(X_fold_val)
        cv_scores.append(f1_score(y_fold_val, y_fold_pred, pos_label="M"))

    cv_scores = np.array(cv_scores)

    return {
        "name": name,
        "model": model,
        "f1": f1_score(y_test, y_pred, pos_label="M"),
        "recall": recall_score(y_test, y_pred, pos_label="M"),
        "precision": precision_score(y_test, y_pred, pos_label="M"),
        "accuracy": accuracy_score(y_test, y_pred),
        "auc": roc_auc_score((y_test == "M").astype(int), y_prob),
        "cv_f1": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "fn": cm[1, 0],
        "fp": cm[0, 1],
        "cm": cm,
        "y_pred": y_pred,
        "y_prob": y_prob
    }

models_to_run = {
    "Logistic Regression (balanced, 21 features)": (
        LogisticRegression(
            class_weight="balanced",
            random_state=42
        ),
        X_train_sc,
        X_test_sc,
        False
    ),

    "Shallow Decision Tree (balanced, 21 features)": (
        DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ),
        X_train_sc,
        X_test_sc,
        False
    ),
    "Linear SVM (balanced, 21 features)": (
        SVC(
            kernel="linear",
            probability=True,
            class_weight="balanced",
            random_state=42
        ),
        X_train_sc,
        X_test_sc,
        False
    ),

    "Explainable Boosting Machine (balanced, 21 features)": (
        ExplainableBoostingClassifier(
            max_rounds=300,
            random_state=42
        ),
        X_train,
        X_test,
        True
    )
}


results_21_balanced = {}

print(f"\n  {'Model':<55} {'F1':>7} {'Recall':>7} {'Prec':>7} {'AUC':>7} {'CV F1':>8} {'FN':>4} {'FP':>4}")
print("  " + "-" * 110)

for name, (model, Xtr, Xte, use_sample_weight) in models_to_run.items():
    result = evaluate_model(name, model, Xtr, Xte, use_sample_weight)
    results_21_balanced[name] = result

    flag = " ✓" if result["f1"] > 0.95 else ""

    print(
        f"  {name:<55} "
        f"{result['f1']:>7.4f} "
        f"{result['recall']:>7.4f} "
        f"{result['precision']:>7.4f} "
        f"{result['auc']:>7.4f} "
        f"{result['cv_f1']:>8.4f} "
        f"{result['fn']:>4} "
        f"{result['fp']:>4}"
        f"{flag}"
    )



  Model                                                        F1  Recall    Prec     AUC    CV F1   FN   FP
  --------------------------------------------------------------------------------------------------------------
  Logistic Regression (balanced, 21 features)              0.9647  0.9762  0.9535  0.9960   0.9579    1    2 ✓
  Shallow Decision Tree (balanced, 21 features)            0.9231  0.8571  1.0000  0.9286   0.9177    6    0
  Linear SVM (balanced, 21 features)                       0.9639  0.9524  0.9756  0.9960   0.9461    2    1 ✓
  Explainable Boosting Machine (balanced, 21 features)     0.9756  0.9524  1.0000  0.9924   0.9578    2    0 ✓


In [6]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

C_values = [1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2]
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55]

results = []

for C in C_values:
    for solver in ["lbfgs", "liblinear"]:

        lr = LogisticRegression(
            C=C,
            solver=solver,
            max_iter=10000,
            class_weight="balanced",
            random_state=42
        )

        lr.fit(X_train_sc, y_train)

        y_prob = lr.predict_proba(X_test_sc)[:, 1]
        auc = roc_auc_score((y_test == "M").astype(int), y_prob)

        for th in thresholds:
            y_pred = (y_prob >= th).astype(int)
            y_pred_lbl = np.where(y_pred == 1, "M", "B")

            cm = confusion_matrix(y_test, y_pred_lbl, labels=["B", "M"])

            cv_scores = []

            for train_idx, val_idx in cv.split(X_train_sc, y_train):
                lr_cv = LogisticRegression(
                    C=C,
                    solver=solver,
                    max_iter=10000,
                    class_weight="balanced",
                    random_state=42
                )

                lr_cv.fit(
                    X_train_sc[train_idx],
                    y_train.iloc[train_idx]
                )

                val_prob = lr_cv.predict_proba(X_train_sc[val_idx])[:, 1]
                val_pred = np.where(val_prob >= th, "M", "B")

                cv_scores.append(
                    f1_score(y_train.iloc[val_idx], val_pred, pos_label="M")
                )

            results.append({
                "C": C,
                "solver": solver,
                "threshold": th,
                "f1": f1_score(y_test, y_pred_lbl, pos_label="M"),
                "recall": recall_score(y_test, y_pred_lbl, pos_label="M"),
                "precision": precision_score(y_test, y_pred_lbl, pos_label="M"),
                "auc": auc,
                "cv_f1": np.mean(cv_scores),
                "cv_std": np.std(cv_scores),
                "fn": cm[1, 0],
                "fp": cm[0, 1]
            })

results_df = pd.DataFrame(results).sort_values(
    ["f1", "recall", "cv_f1"],
    ascending=False
)

print(results_df.head(20).to_string(index=False))

   C    solver  threshold       f1   recall  precision      auc    cv_f1   cv_std  fn  fp
 1.0     lbfgs       0.45 0.964706 0.976190   0.953488 0.996032 0.962338 0.041155   1   2
 1.0 liblinear       0.45 0.964706 0.976190   0.953488 0.996362 0.959639 0.041354   1   2
 1.0     lbfgs       0.50 0.964706 0.976190   0.953488 0.996032 0.957857 0.053088   1   2
 1.0 liblinear       0.50 0.964706 0.976190   0.953488 0.996362 0.957857 0.053088   1   2
 1.0     lbfgs       0.40 0.964706 0.976190   0.953488 0.996032 0.953925 0.037142   1   2
 1.0 liblinear       0.40 0.964706 0.976190   0.953488 0.996362 0.953925 0.037142   1   2
 0.1 liblinear       0.55 0.963855 0.952381   0.975610 0.996693 0.960890 0.038569   2   1
 0.1     lbfgs       0.55 0.963855 0.952381   0.975610 0.996693 0.960338 0.045810   2   1
 1.0     lbfgs       0.55 0.963855 0.952381   0.975610 0.996032 0.951266 0.048095   2   1
 1.0 liblinear       0.55 0.963855 0.952381   0.975610 0.996362 0.951266 0.048095   2   1
 1.0 libli

In [7]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = []

for depth in range(2, 15):
    for leaf in range(1, 10):   

        tree = DecisionTreeClassifier(
            max_depth=depth,
            min_samples_leaf=leaf,
            class_weight="balanced",
            random_state=42
        )

        tree.fit(X_train_sc, y_train)
        y_pred = tree.predict(X_test_sc)
        y_prob = tree.predict_proba(X_test_sc)[:, 1]

        cm = confusion_matrix(y_test, y_pred, labels=["B", "M"])

        cv_scores = []

        for train_idx, val_idx in cv.split(X_train_sc, y_train):
            t = clone(tree)
            t.fit(X_train_sc[train_idx], y_train.iloc[train_idx])
            yv = t.predict(X_train_sc[val_idx])

            cv_scores.append(
                f1_score(y_train.iloc[val_idx], yv, pos_label="M")
            )

        results.append({
            "max_depth": depth,
            "min_samples_leaf": leaf,
            "f1": f1_score(y_test, y_pred, pos_label="M"),
            "recall": recall_score(y_test, y_pred, pos_label="M"),
            "precision": precision_score(y_test, y_pred, pos_label="M"),
            "auc": roc_auc_score((y_test == "M").astype(int), y_prob),
            "cv_f1": np.mean(cv_scores),
            "cv_std": np.std(cv_scores),
            "fn": cm[1, 0],
            "fp": cm[0, 1]
        })

results_df = pd.DataFrame(results)

results_df.sort_values(
    by=["f1", "recall", "cv_f1"],
    ascending=False
).head(20)

,max_depth,min_samples_leaf,f1,recall,precision,auc,cv_f1,cv_std,fn,fp
28,5,2,0.950000,0.904762,1.000000,0.933532,0.925143,0.063036,4,0
27,5,1,0.938272,0.904762,0.974359,0.926257,0.931185,0.062585,4,1
20,4,3,0.938272,0.904762,0.974359,0.926422,0.927569,0.053559,4,1
29,5,3,0.938272,0.904762,0.974359,0.925926,0.926367,0.057449,4,1
47,7,3,0.938272,0.904762,0.974359,0.945271,0.926367,0.057449,4,1
56,8,3,0.938272,0.904762,0.974359,0.945271,0.926367,0.057449,4,1
65,9,3,0.938272,0.904762,0.974359,0.945271,0.926367,0.057449,4,1
74,10,3,0.938272,0.904762,0.974359,0.945271,0.926367,0.057449,4,1
83,11,3,0.938272,0.904762,0.974359,0.945271,0.926367,0.057449,4,1
92,12,3,0.938272,0.904762,0.974359,0.945271,0.926367,0.057449,4,1


In [8]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

C_values = [1e-4, 1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2]
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55]

results = []

for C, th in product(C_values, thresholds):

    svm = SVC(
        kernel="linear",
        C=C,
        probability=True,
        class_weight="balanced",
        random_state=42
    )

    svm.fit(X_train_sc, y_train)

    y_prob = svm.predict_proba(X_test_sc)[:, 1]
    y_pred = np.where(y_prob >= th, "M", "B")

    cm = confusion_matrix(y_test, y_pred, labels=["B", "M"])
    auc = roc_auc_score((y_test == "M").astype(int), y_prob)

    cv_scores = []

    for train_idx, val_idx in cv.split(X_train_sc, y_train):

        svm_cv = SVC(
            kernel="linear",
            C=C,
            probability=True,
            class_weight="balanced",
            random_state=42
        )

        svm_cv.fit(
            X_train_sc[train_idx],
            y_train.iloc[train_idx]
        )

        val_prob = svm_cv.predict_proba(X_train_sc[val_idx])[:, 1]
        val_pred = np.where(val_prob >= th, "M", "B")

        cv_scores.append(
            f1_score(y_train.iloc[val_idx], val_pred, pos_label="M")
        )

    results.append({
        "C": C,
        "threshold": round(th, 2),
        "f1": f1_score(y_test, y_pred, pos_label="M"),
        "recall": recall_score(y_test, y_pred, pos_label="M"),
        "precision": precision_score(y_test, y_pred, pos_label="M"),
        "auc": auc,
        "cv_f1": np.mean(cv_scores),
        "cv_std": np.std(cv_scores),
        "fn": cm[1, 0],
        "fp": cm[0, 1]
    })

results_df = pd.DataFrame(results).sort_values(
    ["fn", "f1", "cv_f1"],
    ascending=[True, False, False]
)

print(results_df.head(20).to_string(index=False))

     C  threshold       f1   recall  precision      auc    cv_f1   cv_std  fn  fp
0.0001       0.30 0.865979 1.000000   0.763636 0.977183 0.644149 0.074308   0  13
0.0100       0.45 0.976190 0.976190   0.976190 0.997354 0.952145 0.041307   1   1
0.0100       0.50 0.976190 0.976190   0.976190 0.997354 0.951416 0.044213   1   1
0.0100       0.40 0.964706 0.976190   0.953488 0.997354 0.955353 0.039021   1   2
0.1000       0.45 0.964706 0.976190   0.953488 0.996362 0.955229 0.036238   1   2
0.0100       0.35 0.964706 0.976190   0.953488 0.997354 0.952512 0.041191   1   2
0.0010       0.35 0.964706 0.976190   0.953488 0.994378 0.925890 0.047132   1   2
0.1000       0.40 0.953488 0.976190   0.931818 0.996362 0.958779 0.037880   1   3
1.0000       0.30 0.953488 0.976190   0.931818 0.996032 0.954242 0.034780   1   3
0.1000       0.35 0.942529 0.976190   0.911111 0.996362 0.956432 0.041906   1   4
0.1000       0.30 0.942529 0.976190   0.911111 0.996362 0.954069 0.034698   1   4
0.0100       0.3

In [9]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
max_rounds_values = [100, 200, 300]
outer_bags_values = [2, 4, 8, 12]
learning_rates    = [0.001, 0.005, 0.01]

results = []

for max_rounds, outer_bags, lr in product(
    max_rounds_values,
    outer_bags_values,
    learning_rates
):

    ebm = ExplainableBoostingClassifier(
        interactions=0,
        max_rounds=max_rounds,
        outer_bags=outer_bags,
        learning_rate=lr,
        random_state=42
    )

    sw = compute_sample_weight("balanced", y=y_train)
    ebm.fit(X_train, y_train, sample_weight=sw)

    y_pred = ebm.predict(X_test)
    y_prob = ebm.predict_proba(X_test)[:, 1]

    cm = confusion_matrix(y_test, y_pred, labels=["B", "M"])

    # Test-set AUC
    y_test_bin = (y_test == "M").astype(int)
    auc = roc_auc_score(y_test_bin, y_prob)

    # 10-fold CV F1
    cv_scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):

        model = clone(ebm)

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        sw_cv = compute_sample_weight("balanced", y=y_tr)

        model.fit(X_tr, y_tr, sample_weight=sw_cv)

        y_val_pred = model.predict(X_val)

        cv_scores.append(
            f1_score(y_val, y_val_pred, pos_label="M")
        )

    results.append({
        "max_rounds": max_rounds,
        "outer_bags": outer_bags,
        "learning_rate": lr,
        "f1": f1_score(y_test, y_pred, pos_label="M"),
        "recall": recall_score(y_test, y_pred, pos_label="M"),
        "precision": precision_score(y_test, y_pred, pos_label="M"),
        "auc": auc,
        "cv_f1": np.mean(cv_scores),
        "cv_std": np.std(cv_scores),
        "fn": cm[1, 0],
        "fp": cm[0, 1]
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    ["fn", "cv_f1", "f1"],
    ascending=[True, False, False]
)

print(results_df.to_string(index=False))

 max_rounds  outer_bags  learning_rate       f1   recall  precision      auc    cv_f1   cv_std  fn  fp
        300           2          0.010 0.975610 0.952381   1.000000 0.990079 0.963598 0.038646   2   0
        300           4          0.010 0.975610 0.952381   1.000000 0.989418 0.963593 0.036340   2   0
        200           8          0.010 0.975610 0.952381   1.000000 0.990741 0.963563 0.030768   2   0
        300           8          0.010 0.975610 0.952381   1.000000 0.988757 0.963563 0.030768   2   0
        300          12          0.005 0.975610 0.952381   1.000000 0.991733 0.963563 0.030768   2   0
        200          12          0.010 0.975610 0.952381   1.000000 0.992394 0.960706 0.028494   2   0
        300          12          0.010 0.975610 0.952381   1.000000 0.989087 0.960706 0.028494   2   0
        200           2          0.010 0.962963 0.928571   1.000000 0.992063 0.963598 0.038646   3   0
        300           8          0.005 0.962963 0.928571   1.000000 0.990

In [10]:
# BEST MODELING BASED ON THE INITIAL PARAMETER OPTIMIZATIONS
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

def evaluate_model(name, model, Xtr, Xte, use_sample_weight=False, threshold=None):

    if use_sample_weight:
        sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
        model.fit(Xtr, y_train, sample_weight=sample_weight)
    else:
        model.fit(Xtr, y_train)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(Xte)[:, 1]
    else:
        y_prob = model.decision_function(Xte)

    if threshold is not None:
        y_pred = np.where(y_prob >= threshold, "M", "B")
    else:
        y_pred = model.predict(Xte)

    cm = confusion_matrix(y_test, y_pred, labels=["B", "M"])

    cv_scores = []

    for train_idx, val_idx in cv.split(Xtr, y_train):
        model_cv = clone(model)

        X_fold_train = Xtr[train_idx] if isinstance(Xtr, np.ndarray) else Xtr.iloc[train_idx]
        X_fold_val = Xtr[val_idx] if isinstance(Xtr, np.ndarray) else Xtr.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        if use_sample_weight:
            fold_weight = compute_sample_weight(class_weight="balanced", y=y_fold_train)
            model_cv.fit(X_fold_train, y_fold_train, sample_weight=fold_weight)
        else:
            model_cv.fit(X_fold_train, y_fold_train)

        if hasattr(model_cv, "predict_proba"):
            y_fold_prob = model_cv.predict_proba(X_fold_val)[:, 1]
        else:
            y_fold_prob = model_cv.decision_function(X_fold_val)

        if threshold is not None:
            y_fold_pred = np.where(y_fold_prob >= threshold, "M", "B")
        else:
            y_fold_pred = model_cv.predict(X_fold_val)

        cv_scores.append(f1_score(y_fold_val, y_fold_pred, pos_label="M"))

    cv_scores = np.array(cv_scores)

    return {
        "name": name,
        "model": model,
        "f1": f1_score(y_test, y_pred, pos_label="M"),
        "recall": recall_score(y_test, y_pred, pos_label="M"),
        "precision": precision_score(y_test, y_pred, pos_label="M"),
        "accuracy": accuracy_score(y_test, y_pred),
        "auc": roc_auc_score((y_test == "M").astype(int), y_prob),
        "cv_f1": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "fn": cm[1, 0],
        "fp": cm[0, 1],
        "cm": cm,
        "y_pred": y_pred,
        "y_prob": y_prob
    }

models_to_run = {
    "Logistic Regression (C=1.0, th=0.45, balanced)": (
        LogisticRegression(
            C=1.0,
            solver="lbfgs",
            max_iter=10000,
            class_weight="balanced",
            random_state=42
        ),
        X_train_sc,
        X_test_sc,
        False,
        0.45
    ),

    "Shallow Decision Tree (depth=5, leaf=2, balanced)": (
        DecisionTreeClassifier(
            max_depth=5,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42
        ),
        X_train_sc,
        X_test_sc,
        False,
        None
    ),
    "Linear SVM (C=0.01, th=0.45, balanced)": (
        SVC(
            kernel="linear",
            C=0.01,
            probability=True,
            class_weight="balanced",
            random_state=42
        ),
        X_train_sc,
        X_test_sc,
        False,
        0.45
    ),

    "Explainable Boosting Machine (bs=2, rs=300, lr=0.010)": (
        ExplainableBoostingClassifier(
            interactions=0,
            outer_bags=2,
            max_rounds=300,
            learning_rate=0.010,
            random_state=42
        ),
        X_train,
        X_test,
        True,
        None
    )
}


results_21_balanced = {}

print(f"\n  {'Model':<55} {'F1':>7} {'Recall':>7} {'Prec':>7} {'AUC':>7} {'CV F1':>8} {'FN':>4} {'FP':>4}")
print("  " + "-" * 100)

for name, (model, Xtr, Xte, use_sample_weight, threshold) in models_to_run.items():
    result = evaluate_model(name, model, Xtr, Xte, use_sample_weight, threshold)
    results_21_balanced[name] = result

    flag = " ✓" if result["f1"] > 0.95 else ""

    print(
        f"  {name:<55} "
        f"{result['f1']:>7.4f} "
        f"{result['recall']:>7.4f} "
        f"{result['precision']:>7.4f} "
        f"{result['auc']:>7.4f} "
        f"{result['cv_f1']:>8.4f} "
        f"{result['fn']:>4} "
        f"{result['fp']:>4}"
        f"{flag}"
    )


  Model                                                        F1  Recall    Prec     AUC    CV F1   FN   FP
  ----------------------------------------------------------------------------------------------------
  Logistic Regression (C=1.0, th=0.45, balanced)           0.9647  0.9762  0.9535  0.9960   0.9623    1    2 ✓
  Shallow Decision Tree (depth=5, leaf=2, balanced)        0.9500  0.9048  1.0000  0.9335   0.9251    4    0
  Linear SVM (C=0.01, th=0.45, balanced)                   0.9762  0.9762  0.9762  0.9974   0.9521    1    1 ✓
  Explainable Boosting Machine (bs=2, rs=300, lr=0.010)    0.9756  0.9524  1.0000  0.9901   0.9636    2    0 ✓
